# Five-epoch multilingual Qwen3 NF4 smoke run
Enable Internet and GPU T4 x2. Seven tiny language samples, 512 training tokens each, five COMPLETE epochs. No step or time cap by default. Search and pilot benchmarks add runtime. New output folder prevents reuse of older experiments.


## 1. Clone or update

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, importlib.metadata
ROOT=Path('/kaggle/working')
REPO=ROOT/'Layer-Merging'
OUTPUT=ROOT/'outputs/smoke_5epoch_multilingual_nf4'
print('Python:',sys.version)
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/amritosanto/Layer-Merging.git',str(REPO)],check=True)
else:
    if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
        raise RuntimeError('Repository has local edits; preserve them before updating')
    subprocess.run(['git','pull','--ff-only'],cwd=REPO,check=True)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Source revision:',REVISION)


## 2. Install compatible binary wheels, preserving CUDA PyTorch

In [ ]:
PACKAGES = ['transformers==4.51.3', 'tokenizers==0.21.4', 'datasets==4.4.1', 'accelerate==1.6.0', 'bitsandbytes==0.48.2', 'huggingface-hub==0.36.0', 'numpy==2.2.6', 'pandas==2.2.3', 'matplotlib==3.10.1', 'safetensors==0.5.3', 'pytest==8.3.5', 'pyarrow==21.0.0', 'python-dateutil==2.9.0.post0', 'dill==0.4.0', 'multiprocess==0.70.18', 'fsspec==2025.9.0']
torch_version=importlib.metadata.version('torch')
constraint=ROOT/'qwen-keep-torch.txt'
constraint.write_text('torch=='+torch_version+'\n')
subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:','-c',str(constraint),*PACKAGES],check=True)
assert importlib.metadata.version('torch')==torch_version
ENV=os.environ.copy()
ENV.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',PYTHONUNBUFFERED='1')


## 3. Detect both T4 GPUs

In [ ]:
subprocess.run(['nvidia-smi'],check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.device_count()>=2, 'Enable T4 x2'; print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],env=ENV,check=True)


## 4. Load and verify five-epoch settings

In [ ]:
config=json.loads((REPO/'config.smoke.json').read_text())
config['output_dir']=str(OUTPUT)
# Optional overrides for a NEW experiment:
# Keep epochs=5 and max_optimizer_steps=0 for five complete passes.
OUTPUT.mkdir(parents=True,exist_ok=True)
CONFIG=ROOT/'qwen-smoke-run.json'
CONFIG.write_text(json.dumps(config,indent=2))
assert config['epochs']==5 and config['max_optimizer_steps']==0
assert config['training_hours']==0, 'Use no time cap for the complete five-epoch run'
print(json.dumps(config,indent=2))
def run_stage(stage):
    subprocess.run([sys.executable,'run.py',stage,'--config',str(CONFIG)],cwd=REPO,env=ENV,check=True)
subprocess.run([sys.executable,'-c','from config import Config; from src.trainer import hardware; import sys; hardware(Config.load(sys.argv[1]))',str(CONFIG)],cwd=REPO,env=ENV,check=True)
# One-command alternative to the following stage cells: run_stage('smoke')


## 5. Prepare small Wiki-40B and FineWeb2 samples

In [ ]:
run_stage('prepare')


## 6. Essential GPU preflight

In [ ]:
run_stage('preflight')


## 7. Train five complete epochs or resume

In [ ]:
run_stage('train')


## 8. CE search and at most three NF4 finalists

In [ ]:
run_stage('search')


## 9. Export only the selected learned NF4 model

In [ ]:
run_stage('export')


## 10. Evaluate original NF4 and learned NF4

In [ ]:
run_stage('benchmark')


## 11. Generate report

In [ ]:
run_stage('report')


## 12. Results and downloads
Save a Kaggle version with outputs to preserve checkpoints, source manifests, and the cached original NF4 baseline for resume. Individual links below avoid duplicating model shards in a ZIP.

In [ ]:
from IPython.display import display,Markdown,FileLink
import csv
display(Markdown((OUTPUT/'REPORT.md').read_text()))
for name in ['accuracy_comparison.csv','per_language_accuracy.csv','answer_option_frequencies.csv']:
    p=OUTPUT/'benchmark_results'/config['evaluation_mode']/name
    with p.open() as stream:display(list(csv.DictReader(stream)))
for path in [OUTPUT/'REPORT.md',OUTPUT/'checkpoints/merger_final.pt',OUTPUT/'rankings/selected_plan.json',OUTPUT/'manifests/language_coverage.json']:
    display(FileLink(str(path)))
for path in sorted((OUTPUT/'model_exports/best_learned_nf4').iterdir()):
    if path.is_file():display(FileLink(str(path)))
